# L3 — Specialist Agents

## What you'll learn

- How to deploy a specialist agent on AgentCore Runtime so the orchestrator can call it
- How a specialist agent reaches its tools through the AgentCore Gateway
- How to register an agent in the Registry so the orchestrator can find it
- How one deploy, test and register flow serves every specialist agent

## Overview

AnyCompany's support system has three specialist agents. Each one handles one kind of question and reaches its tools through the AgentCore Gateway:

- **Order Agent.** Order lookups and a customer's order history. Uses the Order tool.
- **Refund Agent.** Return-policy questions, eligibility checks and refunds. Uses the Refund tools, which read the vector store you set up in L1.
- **Research Agent (optional).** Product reviews, specs and comparisons, with cited sources. Uses the web-search tool on the Gateway.

Each agent goes through the same three steps: **deploy** it to AgentCore Runtime, **test** it with one question, and **register** it in the Registry so the orchestrator can find it. The steps are written once and run for each agent.

## Architecture

![Orchestration layer: the AgentCore Harness with its memory discovers the Order, Refund and Research agents through the Registry; the agents call tools through the AgentCore Gateway, which reaches the order tools and DynamoDB, the refund tools and the knowledge base (Bedrock or Couchbase), and one web-search provider.](images/l3-orchestration-architecture.png)

This notebook deploys the agents shown under Registry in the diagram and registers them there. The Gateway and its tools were added earlier, and the next notebook adds the orchestrator (the Harness).

## Steps

0. Set up: install the packages and load the configuration
1. Define the agents and load the shared steps
2. Order Agent: deploy, test, register
3. Refund Agent: deploy, test, register
4. Research Agent (optional): deploy, test, register
5. Save the agent IDs

## License details
Refer to LICENSE file in the main folder for license details.

## Prerequisites

- L1 ([`1_knowledge_base.ipynb`](../l1-data-knowledge/1_knowledge_base.ipynb)) and L2 ([`1_pluggable_inference_layer.ipynb`](../l2-inference/1_pluggable_inference_layer.ipynb)) are complete.
- [`1_gateway_tools.ipynb`](1_gateway_tools.ipynb) is complete, including your web-search and observability choices.

> **About the starter toolkit:** this workshop uses the final release (`0.3.14`) of `bedrock-agentcore-starter-toolkit`, which is deprecated. It was tested with these notebooks, so it is pinned on purpose. For your own projects, use the [AgentCore CLI](https://www.npmjs.com/package/@aws/agentcore) (`@aws/agentcore`).

## Step 0: Set up

### 0.1 Install the packages

Install the AWS SDK and the AgentCore libraries. Run this once, at the start.

In [ ]:
%pip install --quiet --upgrade boto3 requests bedrock-agentcore==1.14.0 bedrock-agentcore-starter-toolkit==0.3.14

### 0.2 Load the workshop configuration

Read the settings the workshop saved earlier: the Region, your observability choice, and the web-search tool if one is connected.

In [ ]:
import boto3
import os

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")  # Change to your preferred region
SSM_PREFIX = "/anycompany/agentcore"

ssm = boto3.client("ssm", region_name=REGION)
sts = boto3.client("sts", region_name=REGION)


def get_ssm(name, default=None):
    try:
        return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        if default is not None:
            return default
        raise


ACCOUNT_ID = sts.get_caller_identity()["Account"]
REGISTRY_ID = get_ssm(f"{SSM_PREFIX}/registry_id")
GATEWAY_ID = get_ssm(f"{SSM_PREFIX}/gateway_id")
MODEL_ID = get_ssm(f"{SSM_PREFIX}/model_id")  # from the L2 inference layer
ORDERS_TABLE = get_ssm(f"{SSM_PREFIX}/dynamodb_orders_table")
# The L1 knowledge store the Refund tools read: Bedrock Knowledge Base or Couchbase.
KB_PROVIDER = get_ssm(f"{SSM_PREFIX}/kb_provider", default="bedrock")
KB_READY = bool(get_ssm(f"{SSM_PREFIX}/" + ("kb_chunks_keyspace" if KB_PROVIDER == "couchbase" else "kb_id"), default=""))

# The web-search tool is connected in 1_gateway_tools.ipynb (Step 4). It's only
# needed for the optional Research Agent, so a missing one is not an error here.
from helpers import websearch as ws
WEBSEARCH = ws.active(REGION)

# Observability provider (selected in the Partner Selection UI): CloudWatch native, Datadog OTEL-direct,
# or CloudZero (CloudWatch plus the specialists' model-call spans to CloudZero).
from helpers.observability import get_observability_provider
OBS_PROVIDER = get_observability_provider(ssm, SSM_PREFIX)

print(f"Account:       {ACCOUNT_ID}")
print(f"Region:        {REGION}")
print(f"Registry:      {REGISTRY_ID}")
print(f"Gateway:       {GATEWAY_ID}")
print(f"Model:         {MODEL_ID}")
print(f"Orders table:  {ORDERS_TABLE}")
print(f"Knowledge:     {KB_PROVIDER}{'' if KB_READY else ' (not set up)'}")
print(f"Web search:    {WEBSEARCH['label'] if WEBSEARCH['provider'] else '(not connected; the Research Agent will be skipped)'}")
print(f"Observability: {OBS_PROVIDER}")

## Step 1: Define the agents and load the shared steps

Every specialist agent goes through the same three steps: deploy, test and register. Only a few settings differ per agent: its entrypoint, its runtime role, its registry record, and the question used to test it. Those settings are kept together in the cell below. The three steps are written once, in [`helpers/specialists.py`](helpers/specialists.py).

### 1.1 The agents

In [ ]:
# Per-agent settings. Runtime names, registry record names, and SSM keys match
# what later layers (L5, L6, L7) read, so keep them as they are.
AGENTS = {
    "order": {
        "label": "Order Agent",
        "agent_name": "order_agent_a2a",
        "entrypoint": "agents/order_agent_a2a.py",
        "requirements": "agents/requirements_order_a2a.txt",
        "role_param": "order_agent_runtime_role_arn",
        "service_name": "anycompany-order-agent",          # Datadog / CloudZero service name
        "record_name": "order_agent_a2a_record",
        "display_name": "Order Agent (A2A)",
        "description": "Order Agent — handles order lookups and customer order history.",
        "test_prompt": "What is the status of order ORD-10001?",
        "test_as_customer": True,                          # send a Cognito token, as the Harness would
        "arn_param": "order_agent_runtime_arn",
    },
    "refund": {
        "label": "Refund Agent",
        "agent_name": "refund_agent_a2a",
        "entrypoint": "agents/refund_agent_a2a.py",
        "requirements": "agents/requirements_refund_a2a.txt",
        "role_param": "refund_agent_runtime_role_arn",
        "service_name": "anycompany-refund-agent",
        "record_name": "refund_agent_a2a",
        "display_name": "Refund Agent (A2A)",
        "description": "Refund Agent — handles policy queries, eligibility checks, and refund processing.",
        "test_prompt": "What is the refund policy for electronics?",
        "test_as_customer": True,
        "arn_param": "refund_agent_runtime_arn",
    },
    "research": {
        "label": "Research Agent",
        "agent_name": "research_agent_a2a",
        "entrypoint": "agents/research_agent_a2a.py",
        "requirements": "agents/requirements_research_a2a.txt",
        "role_param": "research_agent_runtime_role_arn",
        "service_name": "anycompany-research-agent",
        "record_name": "research_agent_a2a_record",
        "display_name": "Research Agent (A2A)",
        "description": ("Research Agent — live web research for retail products: reviews, "
                        "specifications, comparisons, availability and recent news, with cited sources."),
        "test_prompt": "What do recent reviews say about the battery life of the Google Pixel 9? Cite your sources.",
        "test_as_customer": False,
        "arn_param": "research_agent_runtime_arn",
    },
}

for key, a in AGENTS.items():
    print(f"{a['label']:15s} runtime={a['agent_name']:20s} record={a['record_name']}")

### 1.2 Load the three steps

The code for the three steps is in [`helpers/specialists.py`](helpers/specialists.py), so you don't repeat it for every agent:

- **Deploy** puts the agent on AgentCore Runtime. This takes a few minutes per agent.
- **Test** asks the agent one question and prints its answer.
- **Register** adds the agent to the Registry, so the orchestrator can find it.

Run the cell below to load them.

In [ ]:
from helpers.specialists import deploy_agent, test_agent, register_agent

## Step 2: Order Agent

The Order Agent answers order questions: one order by ID, or all of a customer's orders. It uses the Order tool from `1_gateway_tools.ipynb`, which reads the orders table in DynamoDB.

### 2.1 Deploy the agent

Deploy the agent to AgentCore Runtime. This takes a few minutes.

In [ ]:
ORDER_AGENT_ARN = deploy_agent(AGENTS["order"], REGION, OBS_PROVIDER)

### 2.2 Test the agent

Ask the agent one question and check that it answers.

In [ ]:
test_agent(AGENTS["order"], ORDER_AGENT_ARN, REGION);

### 2.3 Register the agent

Add the agent to the Registry so the orchestrator can find it.

In [ ]:
register_agent(AGENTS["order"], ORDER_AGENT_ARN, REGION)

## Step 3: Refund Agent

The Refund Agent answers return-policy questions, checks whether an order is eligible for a refund, and processes refunds. It uses the Refund tools from `1_gateway_tools.ipynb`, which read policy passages from the vector store you set up in L1 (Bedrock Knowledge Base or Couchbase).

### 3.1 Deploy the agent

Deploy the agent to AgentCore Runtime. This takes a few minutes.

In [ ]:
if not KB_READY:
    print(f"⚠ No {KB_PROVIDER} knowledge base in SSM. Policy answers need the L1 knowledge base; "
          "run l1-data-knowledge/1_knowledge_base.ipynb first.\n")
REFUND_AGENT_ARN = deploy_agent(AGENTS["refund"], REGION, OBS_PROVIDER)

### 3.2 Test the agent

Ask the agent one question and check that it answers.

In [ ]:
test_agent(AGENTS["refund"], REFUND_AGENT_ARN, REGION);

### 3.3 Register the agent

Add the agent to the Registry so the orchestrator can find it.

In [ ]:
register_agent(AGENTS["refund"], REFUND_AGENT_ARN, REGION)

## Step 4: Research Agent (optional)

The Research Agent answers questions that need current information no internal system has, such as product reviews, specs and comparisons, by searching the web and citing its sources. It uses the web-search tool you connected in Step 4 of [`1_gateway_tools.ipynb`](1_gateway_tools.ipynb): AgentCore Web Search, Brave Search or Perplexity Search. The agent code is the same for all three.

If no web-search tool is connected, these cells skip the Research Agent and the rest of the workshop still works. To switch providers later, re-run Step 4 of `1_gateway_tools.ipynb`. The agent picks up the change within about a minute, so it doesn't need to be redeployed.

### 4.1 Deploy the agent

Deploy the agent to AgentCore Runtime. This takes a few minutes.

In [ ]:
RESEARCH_AGENT_ARN = None
if WEBSEARCH["provider"]:
    print(f"Web search: {WEBSEARCH['label']} (gateway target {WEBSEARCH['target']})")
    RESEARCH_AGENT_ARN = deploy_agent(AGENTS["research"], REGION, OBS_PROVIDER)
else:
    print("No web-search tool connected; skipping the Research Agent. "
          "Run Step 4 of 1_gateway_tools.ipynb, then re-run this section to add it.")

### 4.2 Test the agent

Ask the agent a research question and check that it answers. The answer should cite source URLs.

In [ ]:
if RESEARCH_AGENT_ARN:
    test_agent(AGENTS["research"], RESEARCH_AGENT_ARN, REGION)

### 4.3 Register the agent

Add the agent to the Registry so the orchestrator can find it.

In [ ]:
if RESEARCH_AGENT_ARN:
    register_agent(AGENTS["research"], RESEARCH_AGENT_ARN, REGION)

## Step 5: Save the agent IDs

Save each agent's runtime ID for the later layers (L4 to L7). The tools' IDs were saved by [`1_gateway_tools.ipynb`](1_gateway_tools.ipynb).

In [ ]:
deployed = {"order": ORDER_AGENT_ARN, "refund": REFUND_AGENT_ARN, "research": RESEARCH_AGENT_ARN}
for key, arn in deployed.items():
    if not arn:
        continue
    name = f"{SSM_PREFIX}/{AGENTS[key]['arn_param']}"
    ssm.put_parameter(Name=name, Value=arn, Type="SecureString", Overwrite=True)
    print(f"  {name} = {arn}")
print("\nDone.")

## Summary

This notebook deployed the Order, Refund and Research agents (the Research Agent is optional), tested each one, and registered them so the orchestrator can find them. The tools they call were added in [`1_gateway_tools.ipynb`](1_gateway_tools.ipynb).

**Next:** [`3_orchestrator_agent.ipynb`](3_orchestrator_agent.ipynb). Create the orchestrator that finds these agents and delegates each customer request to the right one.

---
## Cleanup

Deletes what this notebook created for every agent that was deployed: its runtime and registry record, its container repository and build project, its log groups, and its saved ID. The runtimes' IAM roles belong to the workshop stack and are removed when it's deleted.

Run this **before** the cleanup in [`1_gateway_tools.ipynb`](1_gateway_tools.ipynb), so the registry records are gone before the registry itself is deleted. Uncomment the cell to run it.

In [ ]:
## === CLEANUP: Delete all resources created in this notebook ===
## Uncomment and run this cell only after you are done with ALL notebooks.
## NOTE: The tools (Lambdas + Gateway targets) are created AND cleaned up in
##       1_gateway_tools.ipynb, not here. Run this BEFORE that cleanup, so the
##       registry records drain before the registry itself is deleted.

# import boto3, os, time
#
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
#
# # (runtime name, registry record name, SSM key for the runtime ARN)
# CLEANUP = [
#     ("order_agent_a2a",    "order_agent_a2a_record",    "order_agent_runtime_arn"),
#     ("refund_agent_a2a",   "refund_agent_a2a",          "refund_agent_runtime_arn"),
#     ("research_agent_a2a", "research_agent_a2a_record", "research_agent_runtime_arn"),
# ]
#
# ssm               = boto3.client("ssm", region_name=REGION)
# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
# agent_registry    = boto3.client("agent-registry-control", region_name=REGION)
# logs_client       = boto3.client("logs", region_name=REGION)
# ecr               = boto3.client("ecr", region_name=REGION)
# codebuild         = boto3.client("codebuild", region_name=REGION)
#
# try:
#     REGISTRY_ID = ssm.get_parameter(Name=f"{SSM_PREFIX}/registry_id")["Parameter"]["Value"]
# except ssm.exceptions.ParameterNotFound:
#     REGISTRY_ID = None
#
# runtime_ids = {}
# for page in agentcore_control.get_paginator("list_agent_runtimes").paginate():
#     for rt in page.get("agentRuntimes", []):
#         runtime_ids[rt.get("agentRuntimeName")] = rt.get("agentRuntimeId")
#
# for runtime_name, record_name, arn_param in CLEANUP:
#     print(f"\n--- {runtime_name} ---")
#
#     # 1. Registry record (must precede the 1_gateway_tools.ipynb registry delete)
#     if REGISTRY_ID:
#         try:
#             for page in agent_registry.get_paginator("list_registry_records").paginate(registryId=REGISTRY_ID):
#                 for rec in page.get("registryRecords", []):
#                     if rec.get("name") == record_name:
#                         agent_registry.delete_registry_record(registryId=REGISTRY_ID, recordId=rec["recordId"])
#                         print(f"Deleted registry record: {record_name}")
#         except Exception as e:
#             print(f"Registry record delete: {e}")
#
#     # 2. Runtime, polled until gone
#     runtime_id = runtime_ids.get(runtime_name)
#     if runtime_id:
#         try:
#             agentcore_control.delete_agent_runtime(agentRuntimeId=runtime_id)
#             print(f"Deleting runtime {runtime_id}...")
#             deadline = time.time() + 300
#             while time.time() < deadline:
#                 try:
#                     agentcore_control.get_agent_runtime(agentRuntimeId=runtime_id); time.sleep(10)
#                 except agentcore_control.exceptions.ResourceNotFoundException:
#                     print("Runtime deleted."); break
#         except Exception as e:
#             print(f"Runtime delete: {e}")
#
#         # 3. Runtime log groups
#         for lg in [f"/aws/vendedlogs/bedrock-agentcore/runtime/APPLICATION_LOGS/{runtime_id}",
#                    f"/aws/vendedlogs/bedrock-agentcore/runtime/USAGE_LOGS/{runtime_id}"]:
#             try:
#                 logs_client.delete_log_group(logGroupName=lg); print(f"Deleted log group: {lg}")
#             except logs_client.exceptions.ResourceNotFoundException:
#                 pass
#     else:
#         print("Runtime not found (not deployed, or already deleted).")
#
#     # 4. ECR repository and CodeBuild project created by the starter toolkit
#     try:
#         ecr.delete_repository(repositoryName=f"bedrock-agentcore-{runtime_name}", force=True)
#         print(f"Deleted ECR repository: bedrock-agentcore-{runtime_name}")
#     except ecr.exceptions.RepositoryNotFoundException:
#         pass
#     try:
#         codebuild.delete_project(name=f"bedrock-agentcore-{runtime_name}-builder")
#         print(f"Deleted CodeBuild project: bedrock-agentcore-{runtime_name}-builder")
#     except Exception as e:
#         print(f"CodeBuild delete: {e}")
#
#     # 5. SSM parameter
#     try:
#         ssm.delete_parameter(Name=f"{SSM_PREFIX}/{arn_param}"); print(f"Deleted SSM parameter: {arn_param}")
#     except ssm.exceptions.ParameterNotFound:
#         pass
#
# print("\nSpecialist agent resources cleaned up.")